# 94 Methodology Update Summary Reference

Reference-only notebook. It records the structural DAM-methodology update that standardized the active stack and retired legacy models from the non-archived notebook flow.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


In [ ]:
summary_rows = [
    {"topic": "Active models", "decision": "naive_previous_week, naive_previous_year, LEAR, XGBoost, Prophet"},
    {"topic": "Prophet entry", "decision": "Prophet starts at FS2 once calendar and holiday structure is active"},
    {"topic": "Legacy removals", "decision": "ARIMA and SARIMA are removed from the active DAM stack and kept only as archived history"},
    {"topic": "FS ladder", "decision": "FS0 benchmarks, FS1 endogenous explicit, FS2 calendar/holiday plus Prophet, FS3 causal exogenous, FS4 advanced engineered finalists"},
    {"topic": "Shortlisting", "decision": "No shortlist after FS1; first fair shortlist after FS2"},
    {"topic": "Tuning cadence", "decision": "FS0 none, FS1 coarse, FS2 serious, FS3 mandatory retuning, FS4 selective finalist retuning"},
    {"topic": "Execution status", "decision": "Notebook flow standardized; heavy reruns remain opt-in per execution notebook"},
]

display(pd.DataFrame(summary_rows))
display(feature_stage_policy_frame())
display(tuning_cadence_frame())
